In [70]:
from enum import StrEnum
from itertools import pairwise
from typing import NamedTuple

import pandas as pd
from pyscipopt import Model, quicksum

# Schichtplanung: Regeln sichtbar machen und Wünsche erfüllen

Acht Mitarbeiter planen eine Woche mit Früh- und Spätschicht. Jede Schicht
braucht genau zwei Mitarbeiter, jeder arbeitet drei bis vier Schichten.
Verfügbarkeiten sind harte Regeln, Schichtwünsche sind weich. Am Ende fällt
ein Mitarbeiter an zwei Tagen aus, und wir planen neu.

Das Beispiel ist bewusst vereinfacht und kein vollständiger Dienstplan.

## Fiktive Daten

Tage, Schichten und Mitarbeiter stehen als Aufzählung. Ein `Einsatz`
verbindet die drei zu einer Einheit: Mitarbeiter × Tag × Schicht.

In [71]:
class Tag(StrEnum):
    MONTAG = "Montag"
    DIENSTAG = "Dienstag"
    MITTWOCH = "Mittwoch"
    DONNERSTAG = "Donnerstag"
    FREITAG = "Freitag"
    SAMSTAG = "Samstag"
    SONNTAG = "Sonntag"


class Schicht(StrEnum):
    FRUEH = "Früh"
    SPAET = "Spät"


class Mitarbeiter(StrEnum):
    ANNA = "Anna"
    BEN = "Ben"
    CLARA = "Clara"
    DAVID = "David"
    EMIL = "Emil"
    FRIDA = "Frida"
    GRETA = "Greta"
    JONAS = "Jonas"


class Einsatz(NamedTuple):
    mitarbeiter: Mitarbeiter
    tag: Tag
    schicht: Schicht

## Hilfsfunktionen für die Ausgabe

`plan_ausgeben` zeigt einen Plan als Tabelle; Zellen, die sich von einem
zweiten Plan unterscheiden, bekommen ein Ausrufezeichen.
`uebersicht_erstellen` zählt Einsätze und erfüllte Wünsche je Mitarbeiter.
`status_beschreiben` übersetzt den Solverstatus in einen lesbaren Text.

In [72]:
def plan_ausgeben(
    plan: set[Einsatz],
    titel: str,
    vergleich: set[Einsatz] | None = None,
) -> None:
    """Gibt einen Wochenplan als Tabelle aus, Änderungen mit Ausrufezeichen."""
    spalten = {}
    for tag in Tag:
        zellen = []
        for mitarbeiter in Mitarbeiter:
            vorher = []
            nachher = []
            for schicht in Schicht:
                einsatz = Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)
                if einsatz in plan:
                    nachher.append(schicht[0])
                if vergleich is not None and einsatz in vergleich:
                    vorher.append(schicht[0])
            zelle = " ".join(nachher) or "–"
            if vergleich is not None and vorher != nachher:
                zelle += "!"
            zellen.append(zelle)
        spalten[tag] = zellen

    print(titel)
    print(pd.DataFrame(spalten, index=list(Mitarbeiter)))
    print()


def uebersicht_erstellen(plan: set[Einsatz]) -> pd.DataFrame:
    """Zählt Einsätze und erfüllte Wünsche je Mitarbeiter."""
    einsaetze_je_mitarbeiter = []
    erfuellte_wuensche_je_mitarbeiter = []
    for mitarbeiter in Mitarbeiter:
        einsaetze = 0
        erfuellte_wuensche = 0
        for tag in Tag:
            for schicht in Schicht:
                einsatz = Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)
                if einsatz in plan:
                    einsaetze += 1
                if einsatz in plan & wuensche:
                    erfuellte_wuensche += 1
        einsaetze_je_mitarbeiter.append(einsaetze)
        erfuellte_wuensche_je_mitarbeiter.append(erfuellte_wuensche)
    return pd.DataFrame(
        {
            "Einsätze": einsaetze_je_mitarbeiter,
            "davon Wunsch": erfuellte_wuensche_je_mitarbeiter,
        },
        index=list(Mitarbeiter),
    )


def status_beschreiben(status: str) -> str:
    """Übersetzt den SCIP-Status in einen lesbaren Text."""
    if status == "optimal":
        return "optimal (Optimalität bewiesen)"
    if status == "timelimit":
        return "zulässig innerhalb des Zeitlimits"
    raise RuntimeError(f"Unerwarteter Solverstatus: {status}.")

## Entscheidungsvariablen

Für jeden möglichen Einsatz gibt es eine binäre Variable. Der Wert 1 bedeutet
„eingeplant“, der Wert 0 bedeutet „nicht eingeplant“.

In [73]:
modell = Model(problemName="schichtplanung")
modell.hideOutput()
modell.setParam(name="limits/time", value=30.0)

# vtype="B" steht für "binary": 0 = nicht eingeplant, 1 = eingeplant.
x = {
    Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht): modell.addVar(
        name=f"x_{mitarbeiter}_{tag}_{schicht}", vtype="B"
    )
    for mitarbeiter in Mitarbeiter
    for tag in Tag
    for schicht in Schicht
}
print(
    f"{len(Mitarbeiter)} Mitarbeiter × {len(Tag)} Tage × "
    f"{len(Schicht)} Schichten = {len(x)} Variablen"
)

8 Mitarbeiter × 7 Tage × 2 Schichten = 112 Variablen


## Regel 1: Jede Schicht ist besetzt

Jede Früh- und Spätschicht braucht genau zwei Mitarbeiter.

In [74]:
benoetigte_mitarbeiter = 2

for tag in Tag:
    for schicht in Schicht:
        modell.addCons(
            cons=quicksum(
                x[Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)]
                for mitarbeiter in Mitarbeiter
            )
            == benoetigte_mitarbeiter,
            name=f"besetzung_{tag}_{schicht}",
        )

## Regel 2: Höchstens eine Schicht pro Tag

Niemand arbeitet am selben Tag in beiden Schichten.

In [75]:
for mitarbeiter in Mitarbeiter:
    for tag in Tag:
        modell.addCons(
            cons=quicksum(
                x[Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)]
                for schicht in Schicht
            )
            <= 1,
            name=f"tageslimit_{mitarbeiter}_{tag}",
        )

## Regel 3: Verfügbarkeit

Nicht verfügbare Einsätze setzen wir auf 0. Wüssten wir eine
Nichtverfügbarkeit schon vor dem Modellaufbau, könnten wir die Variable auch
gleich weglassen. Hier bleibt sie stehen, damit die Regel sichtbar ist.

In [76]:
nicht_verfuegbare_tage = [
    (Mitarbeiter.ANNA, Tag.DIENSTAG),
    (Mitarbeiter.BEN, Tag.MITTWOCH),
    (Mitarbeiter.CLARA, Tag.DIENSTAG),
    (Mitarbeiter.DAVID, Tag.DONNERSTAG),
    (Mitarbeiter.EMIL, Tag.FREITAG),
    (Mitarbeiter.FRIDA, Tag.DONNERSTAG),
    (Mitarbeiter.GRETA, Tag.MITTWOCH),
    (Mitarbeiter.JONAS, Tag.MONTAG),
]

nicht_verfuegbar: set[Einsatz] = {
    Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)
    for mitarbeiter, tag in nicht_verfuegbare_tage
    for schicht in Schicht
}

# Montagfrüh sind bewusst nur drei Mitarbeiter einsetzbar. So bleibt der geänderte
# Plan nach Annas Ausfall zulässig.
montag_frueh = {Mitarbeiter.ANNA, Mitarbeiter.CLARA, Mitarbeiter.GRETA}
montag_spaet = {Mitarbeiter.ANNA, Mitarbeiter.BEN, Mitarbeiter.DAVID}
nicht_verfuegbar |= {
    Einsatz(mitarbeiter=mitarbeiter, tag=Tag.MONTAG, schicht=Schicht.FRUEH)
    for mitarbeiter in set(Mitarbeiter) - montag_frueh
}
nicht_verfuegbar |= {
    Einsatz(mitarbeiter=mitarbeiter, tag=Tag.MONTAG, schicht=Schicht.SPAET)
    for mitarbeiter in set(Mitarbeiter) - montag_spaet
}

# sorted: Damit die Regeln immer in derselben Reihenfolge entstehen und das
# Ergebnis bei jedem Durchlauf gleich ist.
for einsatz in sorted(nicht_verfuegbar):
    modell.addCons(
        cons=x[einsatz] == 0,
        name=f"nicht_verfuegbar_{einsatz.mitarbeiter}_{einsatz.tag}_{einsatz.schicht}",
    )

## Regel 4: Faire Verteilung

Jeder arbeitet drei bis vier Schichten in der Woche. Bei 28 Einsätzen und
acht Mitarbeitern ist das die faire Aufteilung.

In [77]:
min_schichten_pro_woche = 3
max_schichten_pro_woche = 4

for mitarbeiter in Mitarbeiter:
    wochen_einsaetze = quicksum(
        x[Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=schicht)]
        for tag in Tag
        for schicht in Schicht
    )
    modell.addCons(
        cons=wochen_einsaetze >= min_schichten_pro_woche,
        name=f"wochenminimum_{mitarbeiter}",
    )
    modell.addCons(
        cons=wochen_einsaetze <= max_schichten_pro_woche,
        name=f"wochenmaximum_{mitarbeiter}",
    )

## Regel 5: Keine Spätschicht vor einer Frühschicht

Nach einer Spätschicht darf am nächsten Tag keine Frühschicht folgen.

In [78]:
for mitarbeiter in Mitarbeiter:
    for tag, naechster_tag in pairwise(Tag):
        spaetschicht = x[
            Einsatz(mitarbeiter=mitarbeiter, tag=tag, schicht=Schicht.SPAET)
        ]
        fruehschicht = x[
            Einsatz(mitarbeiter=mitarbeiter, tag=naechster_tag, schicht=Schicht.FRUEH)
        ]
        modell.addCons(
            cons=spaetschicht + fruehschicht <= 1,
            name=f"keine_spaet_frueh_folge_{mitarbeiter}_{tag}",
        )

## Zielfunktion: möglichst viele Wünsche erfüllen

Die Gesamtzahl der Einsätze ist bei exakter Besetzung immer 28 und deshalb
kein sinnvolles Ziel. Stattdessen maximieren wir erfüllte Schichtwünsche.

In [79]:
wuensche: set[Einsatz] = {
    Einsatz(mitarbeiter=Mitarbeiter.ANNA, tag=Tag.MONTAG, schicht=Schicht.FRUEH),
    Einsatz(mitarbeiter=Mitarbeiter.ANNA, tag=Tag.DONNERSTAG, schicht=Schicht.SPAET),
    Einsatz(mitarbeiter=Mitarbeiter.BEN, tag=Tag.MONTAG, schicht=Schicht.SPAET),
    Einsatz(mitarbeiter=Mitarbeiter.CLARA, tag=Tag.MITTWOCH, schicht=Schicht.FRUEH),
    Einsatz(mitarbeiter=Mitarbeiter.DAVID, tag=Tag.FREITAG, schicht=Schicht.SPAET),
    Einsatz(mitarbeiter=Mitarbeiter.EMIL, tag=Tag.DIENSTAG, schicht=Schicht.FRUEH),
    Einsatz(mitarbeiter=Mitarbeiter.FRIDA, tag=Tag.SAMSTAG, schicht=Schicht.FRUEH),
    Einsatz(mitarbeiter=Mitarbeiter.GRETA, tag=Tag.SONNTAG, schicht=Schicht.SPAET),
    Einsatz(mitarbeiter=Mitarbeiter.JONAS, tag=Tag.FREITAG, schicht=Schicht.FRUEH),
}

modell.setObjective(
    expr=quicksum(x[einsatz] for einsatz in sorted(wuensche)), sense="maximize"
)

## Optimieren

Eine Lösung lesen wir nur aus, wenn SCIP eine zulässige Lösung gefunden hat.

In [80]:
modell.optimize()
status = str(modell.getStatus())
if modell.getNSols() == 0:
    raise RuntimeError(f"Keine zulässige Lösung gefunden (Status: {status}).")
status_text = status_beschreiben(status)

loesung = modell.getBestSol()
plan = {
    einsatz
    for einsatz, variable in x.items()
    if modell.getSolVal(sol=loesung, expr=variable) > 0.5
}
print(f"Solverstatus: {status_text}")
print(f"Erfüllte Wünsche: {len(plan & wuensche)} von {len(wuensche)}")

Solverstatus: optimal (Optimalität bewiesen)
Erfüllte Wünsche: 9 von 9


## Kennzahlen zum Ausgangsplan

Erst der Plan als Tabelle, dann die Einsätze und erfüllten Wünsche je
Mitarbeiter.

In [81]:
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

uebersicht = uebersicht_erstellen(plan=plan)

plan_ausgeben(plan=plan, titel="AUSGANGSPLAN")
print("\nÜBERSICHT")
print(uebersicht)

AUSGANGSPLAN
      Montag Dienstag Mittwoch Donnerstag Freitag Samstag Sonntag
Anna       F        –        F          S       –       F       –
Ben        S        –        –          –       S       S       S
Clara      –        –        F          F       –       –       F
David      S        –        –          –       S       S       –
Emil       –        F        S          S       –       –       –
Frida      –        S        –          –       F       F       F
Greta      F        F        –          F       –       –       S
Jonas      –        S        S          –       F       –       –


ÜBERSICHT
       Einsätze  davon Wunsch
Anna          4             2
Ben           4             1
Clara         3             1
David         3             1
Emil          3             1
Frida         4             1
Greta         4             1
Jonas         3             1


## Änderungsszenario: Anna kann an zwei Tagen nicht

Wir verwenden dasselbe Modell weiter, sperren Annas Montag und Donnerstag und
passen die Zielfunktion an: Beim Neuplanen sollen möglichst viele Einsätze
aus dem Ausgangsplan erhalten bleiben.

In [82]:
# SCIP muss den transformierten Zustand freigeben, bevor neue Regeln dazukommen.
modell.freeTransform()

for tag in (Tag.MONTAG, Tag.DONNERSTAG):
    for schicht in Schicht:
        einsatz = Einsatz(mitarbeiter=Mitarbeiter.ANNA, tag=tag, schicht=schicht)
        modell.addCons(
            cons=x[einsatz] == 0,
            name=f"anna_nicht_{einsatz.mitarbeiter}_{einsatz.tag}_{einsatz.schicht}",
        )

# Beim Neuplanen sind Wünsche wichtiger als Änderungen: Ein zusätzlicher Wunsch
# wiegt mehr als alle Einsätze zusammen. Bei gleicher Wunschzahl bleiben
# möglichst viele Einsätze aus dem Ausgangsplan erhalten.
modell.setObjective(
    expr=(len(plan) + 1) * quicksum(x[einsatz] for einsatz in sorted(wuensche))
    + quicksum(x[einsatz] for einsatz in sorted(plan)),
    sense="maximize",
)

modell.optimize()
status_neu = str(modell.getStatus())
status_text_neu = status_beschreiben(status_neu)
loesung_neu = modell.getBestSol()
plan_neu = {
    einsatz
    for einsatz, variable in x.items()
    if modell.getSolVal(sol=loesung_neu, expr=variable) > 0.5
}

plan_ausgeben(plan=plan_neu, titel="NEUER PLAN")
uebersicht_neu = uebersicht_erstellen(plan=plan_neu)
print("\nÜBERSICHT NEUER PLAN")
print(uebersicht_neu)

NEUER PLAN
      Montag Dienstag Mittwoch Donnerstag Freitag Samstag Sonntag
Anna       –        –        F          –       –       F       S
Ben        S        –        –          S       S       S       –
Clara      F        –        F          F       –       –       F
David      S        –        –          –       S       S       –
Emil       –        F        S          S       –       –       –
Frida      –        S        –          –       F       F       F
Greta      F        F        –          F       –       –       S
Jonas      –        S        S          –       F       –       –


ÜBERSICHT NEUER PLAN
       Einsätze  davon Wunsch
Anna          3             0
Ben           4             1
Clara         4             1
David         3             1
Emil          3             1
Frida         4             1
Greta         4             1
Jonas         3             1


## Was hat sich geändert?

Verglichen werden die tatsächlich extrahierten Einsätze. Beim Neuplanen zählt
zuerst die Zahl der erfüllten Wünsche, danach möglichst wenige Änderungen.

In [83]:
entfallene_einsaetze = plan - plan_neu
neue_einsaetze = plan_neu - plan
vergleich = pd.DataFrame(
    {
        "Ausgangsplan": [status_text, len(plan & wuensche), len(plan)],
        "Neuer Plan": [
            status_text_neu,
            len(plan_neu & wuensche),
            len(plan_neu),
        ],
    },
    index=("Solverstatus", "Erfüllte Wünsche", "Einsätze"),
)

print(f"Entfallene Einsätze: {len(entfallene_einsaetze)}")
print(f"Neue Einsätze: {len(neue_einsaetze)}")
print("\nVERGLEICH")
print(vergleich)

Entfallene Einsätze: 3
Neue Einsätze: 3

VERGLEICH
                                    Ausgangsplan                      Neuer Plan
Solverstatus      optimal (Optimalität bewiesen)  optimal (Optimalität bewiesen)
Erfüllte Wünsche                               9                               7
Einsätze                                      28                              28


## Vorher und nachher

Ausgangsplan und neuer Plan.

In [84]:
plan_ausgeben(plan=plan, titel="AUSGANGSPLAN")
plan_ausgeben(plan=plan_neu, titel="NEUER PLAN")

AUSGANGSPLAN
      Montag Dienstag Mittwoch Donnerstag Freitag Samstag Sonntag
Anna       F        –        F          S       –       F       –
Ben        S        –        –          –       S       S       S
Clara      –        –        F          F       –       –       F
David      S        –        –          –       S       S       –
Emil       –        F        S          S       –       –       –
Frida      –        S        –          –       F       F       F
Greta      F        F        –          F       –       –       S
Jonas      –        S        S          –       F       –       –

NEUER PLAN
      Montag Dienstag Mittwoch Donnerstag Freitag Samstag Sonntag
Anna       –        –        F          –       –       F       S
Ben        S        –        –          S       S       S       –
Clara      F        –        F          F       –       –       F
David      S        –        –          –       S       S       –
Emil       –        F        S          S       –  

## Unterschied

Der neue Plan mit Ausrufezeichen an jeder geänderten Zelle.

In [85]:
plan_ausgeben(
    plan=plan_neu,
    titel="NEUER PLAN (Ausrufezeichen = geändert)",
    vergleich=plan,
)

NEUER PLAN (Ausrufezeichen = geändert)
      Montag Dienstag Mittwoch Donnerstag Freitag Samstag Sonntag
Anna      –!        –        F         –!       –       F      S!
Ben        S        –        –         S!       S       S      –!
Clara     F!        –        F          F       –       –       F
David      S        –        –          –       S       S       –
Emil       –        F        S          S       –       –       –
Frida      –        S        –          –       F       F       F
Greta      F        F        –          F       –       –       S
Jonas      –        S        S          –       F       –       –



## Faire Verteilung prüfen

Einsätze und erfüllte Wünsche je Mitarbeiter, vor und nach der Neuplanung.

In [ ]:
vergleich_uebersicht = pd.DataFrame(
    {
        "Einsätze vorher": uebersicht["Einsätze"],
        "Einsätze nachher": uebersicht_neu["Einsätze"],
        "Wünsche vorher": uebersicht["davon Wunsch"],
        "Wünsche nachher": uebersicht_neu["davon Wunsch"],
    }
)

print("FAIRE VERTEILUNG VORHER / NACHHER")
print(vergleich_uebersicht)